# Gold Layer – Product Dimension

This notebook builds the product dimension table by integrating product and category data from multiple Silver layer datasets.

## Build Product Dimension

Join product and category datasets to create a unified product dimension and retain only current product records.

In [0]:
%sql
select * from workspace.silver.crm_prd_info

In [0]:
df =  spark.sql( """
SELECT 
    ROW_NUMBER() OVER(ORDER BY pi.start_date  ,pi.product_id ) AS product_key , 
    pi.prd_key AS product_number, 
    pi.product_id , 
    pi.product_number AS product_category ,  
    pi.product_name , 
    pi.category_id , 
    pc.category , 
    pc.subcategory ,
    pc.maintenance_flag , 
    pi.product_cost , 
    pi.product_line,
    pi.start_date 
   
FROM workspace.silver.crm_prd_info pi
LEFT JOIN workspace.silver.erp_product_category pc
ON pi.product_number = pc.category_id
WHERE pi.end_date IS NULL --Filter out all histroical data
"""
)

## Preview Dimension Data

Review the product dimension before loading it into the Gold layer.

In [0]:
df.limit(10).display()

## Load to Gold Layer

Write the product dimension table to the Gold layer.

In [0]:
df.write.mode('overwrite').saveAsTable('workspace.gold.dim_products')

## Verify Data Load

Validate that the product dimension has been successfully loaded into the Gold layer.

In [0]:
%sql
SELECT *
FROM workspace.gold.dim_products LIMIT 10